# From RAG to Agentic RAG — Results Analysis

This notebook provides an interactive analysis of the three RAG systems evaluated on HotpotQA and MuSiQue.

**Prerequisites**: Run the full experiment pipeline first.
```bash
bash experiments/run_all.sh
```

**Sections**
1. Load results
2. Overall accuracy comparison
3. Breakdown by dataset
4. Breakdown by hop count (MuSiQue)
5. Efficiency analysis (tokens / latency)
6. Retrieval success rate
7. Error analysis & qualitative examples
8. Statistical significance

In [ ]:
import json
import os
import sys
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Project root on path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.evaluation.metrics import normalize, exact_match, token_f1

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

PALETTE = {"naive_rag": "#4C72B0", "iterative_rag": "#55A868", "agentic_rag": "#C44E52"}
LABELS  = {"naive_rag": "Naive RAG", "iterative_rag": "Iterative RAG", "agentic_rag": "Agentic RAG"}
RESULTS_DIR = "../results"

## 1. Load Results

In [ ]:
systems = ["naive_rag", "iterative_rag", "agentic_rag"]
raw: dict[str, list] = {}

for sys_name in systems:
    fpath = os.path.join(RESULTS_DIR, f"{sys_name}.json")
    with open(fpath) as f:
        raw[sys_name] = json.load(f)
    print(f"{LABELS[sys_name]:20s}: {len(raw[sys_name]):,} records")

# Build a flat DataFrame per system
def to_df(results: list, sys_name: str) -> pd.DataFrame:
    rows = []
    for r in results:
        pred  = r.get("predicted_answer", "")
        golds = r.get("all_answers") or [r.get("gold_answer", "")]
        rows.append({
            "system":            sys_name,
            "id":                r["id"],
            "dataset":           r["dataset"],
            "n_hops":            r["n_hops"],
            "question":          r["question"],
            "gold_answer":       r["gold_answer"],
            "predicted_answer":  pred,
            "em":                exact_match(pred, golds),
            "f1":                token_f1(pred, golds),
            "retrieval_steps":   r.get("retrieval_steps", 0),
            "tokens_total":      r.get("tokens_total", 0),
            "latency_s":         r.get("latency_s", 0.0),
            "retrieval_success": int(any(p["is_gold"] for p in r.get("retrieved_passages", []))),
        })
    return pd.DataFrame(rows)

dfs = {s: to_df(raw[s], s) for s in systems}
df  = pd.concat(dfs.values(), ignore_index=True)
print(f"\nCombined DataFrame: {df.shape}")
df.head(3)

## 2. Overall Accuracy Comparison

In [ ]:
overall = (
    df.groupby("system")[["em", "f1", "retrieval_success"]]
    .mean()
    .loc[systems]   # keep display order
    .mul(100)
    .round(2)
)
overall.index = [LABELS[s] for s in systems]
overall.columns = ["EM (%)", "Token F1 (%)", "Retrieval Success (%)"]
print(overall.to_string())
overall

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ["em", "f1"]
x = np.arange(len(metrics))
width = 0.28

for i, sys_name in enumerate(systems):
    vals = [dfs[sys_name][m].mean() * 100 for m in metrics]
    bars = ax.bar(x + i * width, vals, width,
                  label=LABELS[sys_name], color=PALETTE[sys_name], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.1f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(["Exact Match", "Token F1"])
ax.set_ylabel("Score (%)")
ax.set_title("Overall Accuracy — All Questions")
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Breakdown by Dataset

In [ ]:
by_ds = (
    df.groupby(["system", "dataset"])[["em", "f1"]]
    .mean().mul(100).round(2)
    .reset_index()
)
by_ds["system"] = by_ds["system"].map(LABELS)
by_ds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
datasets = ["hotpotqa", "musique"]
ds_labels = ["HotpotQA (2-hop)", "MuSiQue (2–4-hop)"]
x = np.arange(len(datasets))
width = 0.28

for ax, metric, title in zip(axes, ["em", "f1"], ["Exact Match (%)", "Token F1 (%)"]):
    for i, sys_name in enumerate(systems):
        vals = [dfs[sys_name][dfs[sys_name].dataset == ds][metric].mean() * 100
                for ds in datasets]
        bars = ax.bar(x + i * width, vals, width,
                      label=LABELS[sys_name], color=PALETTE[sys_name], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x + width)
    ax.set_xticklabels(ds_labels)
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_ylim(0, 100)
    ax.legend(fontsize=8)

fig.suptitle("Accuracy by Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Breakdown by Hop Count (MuSiQue)

In [ ]:
musique_df = df[df.dataset == "musique"]
by_hops = (
    musique_df.groupby(["system", "n_hops"])["em"]
    .mean().mul(100).round(2)
    .reset_index()
)
by_hops["system"] = by_hops["system"].map(LABELS)
by_hops.pivot(index="n_hops", columns="system", values="em")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
hops = sorted(musique_df.n_hops.unique())

for sys_name in systems:
    sub = musique_df[musique_df.system == sys_name]
    em_by_hop = [sub[sub.n_hops == h]["em"].mean() * 100 for h in hops]
    ax.plot([str(h) for h in hops], em_by_hop,
            marker="o", linewidth=2.2, markersize=8,
            label=LABELS[sys_name], color=PALETTE[sys_name])
    for h, v in zip(hops, em_by_hop):
        ax.annotate(f"{v:.1f}", xy=(str(h), v), xytext=(5, 4),
                    textcoords="offset points", fontsize=8.5)

ax.set_xlabel("Number of Reasoning Hops")
ax.set_ylabel("Exact Match (%)")
ax.set_title("EM by Hop Count — MuSiQue")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Efficiency Analysis

In [ ]:
eff = (
    df.groupby("system")[["retrieval_steps", "tokens_total", "latency_s"]]
    .mean()
    .loc[systems]
    .round(2)
)
eff.index = [LABELS[s] for s in systems]
eff.columns = ["Avg Retrieval Steps", "Avg Tokens", "Avg Latency (s)"]
eff

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
panels = [
    ("retrieval_steps", "Avg Retrieval Steps",  "Steps"),
    ("tokens_total",    "Avg Tokens / Question", "Tokens"),
    ("latency_s",       "Avg Latency (s)",       "Seconds"),
]
colors = [PALETTE[s] for s in systems]

for ax, (col, title, ylabel) in zip(axes, panels):
    vals = [dfs[s][col].mean() for s in systems]
    bars = ax.bar([LABELS[s] for s in systems], vals, color=colors, alpha=0.85)
    for bar, v in zip(bars, vals):
        fmt = f"{v:.0f}" if col == "tokens_total" else f"{v:.2f}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                fmt, ha="center", va="bottom", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticklabels([LABELS[s] for s in systems], rotation=12, ha="right")

fig.suptitle("Efficiency Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy vs Cost scatter (EM vs avg tokens)
fig, ax = plt.subplots(figsize=(6, 5))
for sys_name in systems:
    em  = dfs[sys_name]["em"].mean() * 100
    tok = dfs[sys_name]["tokens_total"].mean()
    ax.scatter(tok, em, s=180, color=PALETTE[sys_name],
               label=LABELS[sys_name], zorder=3)
    ax.annotate(LABELS[sys_name], xy=(tok, em), xytext=(8, 4),
                textcoords="offset points", fontsize=9)

ax.set_xlabel("Avg Tokens / Question")
ax.set_ylabel("Exact Match (%)")
ax.set_title("Accuracy–Cost Trade-off")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Retrieval Success Rate

In [ ]:
ret_success = (
    df.groupby(["system", "dataset"])["retrieval_success"]
    .mean().mul(100).round(2)
    .unstack("dataset")
    .loc[systems]
)
ret_success.index = [LABELS[s] for s in systems]
ret_success.columns.name = None
print("Retrieval Success Rate (%) — fraction of questions where ≥1 gold passage was retrieved")
ret_success

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(systems))
width = 0.35

for j, ds in enumerate(["hotpotqa", "musique"]):
    vals = [dfs[s][dfs[s].dataset == ds]["retrieval_success"].mean() * 100
            for s in systems]
    offset = (j - 0.5) * width
    bars = ax.bar(x + offset, vals, width,
                  label=ds.capitalize(), alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.1f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([LABELS[s] for s in systems])
ax.set_ylabel("Success Rate (%)")
ax.set_title("Retrieval Success Rate\n(≥1 Gold Passage Retrieved)")
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Error Analysis & Qualitative Examples

In [ ]:
def classify_error(row) -> str:
    """Categorise failures into four buckets."""
    if row["em"] == 1:
        return "correct"
    if row["retrieval_success"] == 0:
        return "retrieval_miss"        # gold passage never retrieved
    if token_f1(row["predicted_answer"], [row["gold_answer"]]) > 0.3:
        return "partial_answer"        # retrieved ok, answer partially right
    return "generation_error"          # retrieved ok, but wrong answer

for sys_name in systems:
    dfs[sys_name]["error_type"] = dfs[sys_name].apply(classify_error, axis=1)
df["error_type"] = df.apply(classify_error, axis=1)

In [ ]:
error_dist = (
    df.groupby(["system", "error_type"]).size()
    .unstack("error_type", fill_value=0)
    .loc[systems]
)
error_dist.index = [LABELS[s] for s in systems]
error_pct = error_dist.div(error_dist.sum(axis=1), axis=0).mul(100).round(1)
print("Error distribution (%)")
error_pct

In [ ]:
error_order   = ["correct", "partial_answer", "generation_error", "retrieval_miss"]
error_colors  = ["#55A868", "#8172B3", "#C44E52", "#937860"]
error_order   = [c for c in error_order if c in error_pct.columns]

fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(systems))
x = np.arange(len(systems))

for col, color in zip(error_order, error_colors):
    vals = error_pct[col].values if col in error_pct.columns else np.zeros(len(systems))
    ax.bar(x, vals, bottom=bottom, label=col.replace("_", " ").title(),
           color=color, alpha=0.85)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([LABELS[s] for s in systems])
ax.set_ylabel("Proportion (%)")
ax.set_title("Error Type Distribution")
ax.set_ylim(0, 110)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Qualitative: show 5 cases where Agentic RAG succeeded but Naive RAG failed
naive_fail  = set(dfs["naive_rag"]  [dfs["naive_rag"]["em"]   == 0]["id"])
agent_win   = set(dfs["agentic_rag"][dfs["agentic_rag"]["em"] == 1]["id"])
interesting = list(naive_fail & agent_win)[:5]

print(f"Questions where Naive RAG failed but Agentic RAG succeeded: "
      f"{len(naive_fail & agent_win)}")
print("─" * 70)

for qid in interesting:
    naive_row  = dfs["naive_rag"] [dfs["naive_rag"]["id"]   == qid].iloc[0]
    agent_row  = dfs["agentic_rag"][dfs["agentic_rag"]["id"] == qid].iloc[0]
    print(f"Q:     {naive_row['question']}")
    print(f"Gold:  {naive_row['gold_answer']}")
    print(f"Naive: {naive_row['predicted_answer']}")
    print(f"Agent: {agent_row['predicted_answer']}")
    print("─" * 70)

## 8. Statistical Significance (Paired t-test on F1)

In [ ]:
from scipy import stats

pairs = [
    ("naive_rag",     "iterative_rag"),
    ("naive_rag",     "agentic_rag"),
    ("iterative_rag", "agentic_rag"),
]

# Align on shared question IDs
common_ids = set.intersection(*[set(dfs[s]["id"]) for s in systems])
print(f"Questions shared across all systems: {len(common_ids)}\n")

rows = []
for a, b in pairs:
    fa = dfs[a][dfs[a]["id"].isin(common_ids)].set_index("id")["f1"]
    fb = dfs[b][dfs[b]["id"].isin(common_ids)].set_index("id")["f1"]
    fa, fb = fa.align(fb, join="inner")
    t_stat, p_val = stats.ttest_rel(fb, fa)   # b - a (positive = b is better)
    rows.append({
        "System A": LABELS[a],
        "System B": LABELS[b],
        "Mean F1 A": f"{fa.mean()*100:.2f}%",
        "Mean F1 B": f"{fb.mean()*100:.2f}%",
        "t-stat":    round(t_stat, 3),
        "p-value":   round(p_val, 4),
        "Sig. (p<0.05)": "✓" if p_val < 0.05 else "✗",
    })

pd.DataFrame(rows)